<center><h1>The Annotated Transformer (2025)</h1> </center>


<center>
<p><a href="https://arxiv.org/abs/1706.03762">Attention is All You Need
</a></p>
</center>




from IPython.display import Image
Image(filename='images/paper.png', width=800)

This an updated version of the [The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/), extending the original implementation to support English-to-Chinese translation using a custom-trained modern tokenizer (e.g., RoBERTa).  

This version also replaces Altair with Plotly for visualization and enhances several components, including data loading, batching and data collation. 

Additionally, fixes and improvements are applied to the attention visualization.

<br>



References:
* *[v2022: Austin Huang, Suraj Subramanian, Jonathan Sum, Khalid Almubarak,
   and Stella Biderman]((https://nlp.seas.harvard.edu/annotated-transformer/)).*
* *[Original: Sasha Rush](https://nlp.seas.harvard.edu/2018/04/03/attention.html).*

<br>


# Table of Contents
<ul>

<li><a href="#prelims">Preliminaries</a></li>

<li><a href="#background">Background</a></li>

<li><a href="#part-1-model-architecture">Part 1: Model Architecture</a></li><ul>
<li><a href="#model-architecture">Model Architecture</a></li>
<li><a href="#positional-encoding">Positional Encoding</a></li>
<li><a href="#embeddings-and-softmax">Embeddings</a></li>
<li><a href="#layernorm">LayerNorm</a></li>
<li><a href="#position-wise-feed-forward-networks">Position-wise Feed-Forward
Networks</a></li>
<li><a href="#attention">Attention</a></li>
<li><a href="#encoder-and-decoder-stacks">Encoder and Decoder Stacks</a></li>
<li><a href="#transformer">Transformer Model</a></li>
<li><a href="#full-model">Create Full Model</a></li>
<li><a href="#inference">Inference Test</a></li>
</ul></li>

<li><a href="#part-2-model-training">Part 2: Preparation for Training</a></li><ul>
<li><a href="#batches-and-masking">Batching and Masking</a></li>
<li><a href="#optimizer">Optimizer and Scheduler</a></li>
<li><a href="#training-loop">Training Loop</a></li>
</ul></li>

<li><a href="#part-3-toy-example">Part 3: Toy Training Example</a></li><ul>
<li><a href="#synthetic-data">Synthetic Data</a></li>
<li><a href="#loss-computation">Loss Computation</a></li>
<li><a href="#greedy-decoding">Greedy Decoding</a></li>
<li><a href="#train-loop">Training Loop</a></li>
<li><a href="#train-model">Train the Simple Model</a></li>
</ul></li>

<li><a href="#part-4-a-real-world-example">Part 4: A Real World Example</a></li>
<ul>
<li><a href="#data-loading">Data Loading</a></li>
<li><a href="#iterators">Train Tokenizer</a></li>
<li><a href="#tokenize-data">Tokenize Data</a></li>
<li><a href="#pad-sequence">Pad Sequence Examples</a></li>
<li><a href="#datacollator">Data Collator and Dataloader</a></li>
<li><a href="#training-the-system">Train the System</a></li>
<li><a href="#greedy-decoding">Greedy Decoding and Check Results</a></li>
</ul></li>

<li><a href="#results">Part 5: Attention Visualization</a></li><ul>
<li><a href="#one-example">One example from eval dataset</a></li>
<li><a href="#encoder-self-attention">Encoder Visualization: Self Attention</a></li>
<li><a href="#decoder-self-attention">Decoder Visualization [greedy decoding]: Self Attention</a></li>
<li><a href="#decoder-cross-attention">Decoder Visualization [greedy decoding]: Cross Attention</a></li>
<li><a href="#decoder-self-attention">Decoder Visualization [teacher force]: Self Attention</a></li>
<li><a href="#decoder-cross-attention">Decoder Visualization [teacher force]: Cross Attention</a></li>
</ul></li>

<li><a href="#conclusion">Conclusion</a></li>
</ul>

In this tutorial, I changed the order of the components compared to the previous one. The model architecture comes first, followed by positional encoding, embedding, the feed-forward network, attention, the encoder, and the decoder. 

Finally, the full model is created, and a toy example and a real world example are given.

<br>

# Preliminaries

This tutorial is tested in:
* python=3.12.7
* CUDA=11.8

In [2]:
## Core python packages

# pandas==2.2.3
# datasets==3.0.1
# plotly==5.24.1
# torch==2.4.1
# transformers==4.45.2

# GPUtil==1.4.0

In [3]:
# !pip install GPUtil datasets==3.0.1

In [4]:
# Install torch 2.4.1 when necessary
# !pip install torch==2.4.1 transformers==4.45.2

In [5]:
import os
import math
import copy
import time
from tqdm import tqdm
from dataclasses import dataclass
from typing import Union, Optional, List, Dict, Any


import GPUtil
import plotly
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import LambdaLR
from torch.nn.utils.rnn import pad_sequence

import torch.distributed as dist
import torch.multiprocessing as mp
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP

import datasets
from transformers import AutoTokenizer, PreTrainedTokenizerBase

# import warnings
# warnings.filterwarnings("ignore")

torch.cuda.is_available()

# If cuda is available
if torch.cuda.is_available():
    GPUtil.showUtilization()

<br>

# Background

The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring significantly
less time to train. Our model achieves 28.4 BLEU on the WMT 2014 Englishto-
German translation task, improving over the existing best results, including
ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task,
our model establishes a new single-model state-of-the-art BLEU score of 41.0 after
training for 3.5 days on eight GPUs, a small fraction of the training costs of the
best models from the literature.


The goal of reducing sequential computation also forms the
foundation of the Extended Neural GPU, ByteNet and ConvS2S, all of
which use convolutional neural networks as basic building block,
computing hidden representations in parallel for all input and
output positions. In these models, the number of operations required
to relate signals from two arbitrary input or output positions grows
in the distance between positions, linearly for ConvS2S and
logarithmically for ByteNet. This makes it more difficult to learn
dependencies between distant positions. In the Transformer this is
reduced to a constant number of operations, albeit at the cost of
reduced effective resolution due to averaging attention-weighted
positions, an effect we counteract with Multi-Head Attention.

Self-attention, sometimes called intra-attention is an attention
mechanism relating different positions of a single sequence in order
to compute a representation of the sequence. Self-attention has been
used successfully in a variety of tasks including reading
comprehension, abstractive summarization, textual entailment and
learning task-independent sentence representations. 

End-to-end memory networks are based on a recurrent attention mechanism instead
of sequencealigned recurrence and have been shown to perform well on
simple-language question answering and language modeling tasks.

To the best of our knowledge, however, the Transformer is the first
transduction model relying entirely on self-attention to compute
representations of its input and output without using sequence
aligned RNNs or convolution.

<br>

# Part 1: Model Architecture

## Model Architecture


Most competitive neural sequence transduction models have an
encoder-decoder structure. Here, the encoder maps an
input sequence of symbol representations $(x_1, ..., x_n)$ to a
sequence of continuous representations $\mathbf{z} = (z_1, ...,
z_n)$. Given $\mathbf{z}$, the decoder then generates an output
sequence $(y_1,...,y_m)$ of symbols one element at a time. At each
step the model is auto-regressive, consuming the previously
generated symbols as additional input when generating the next.


The Transformer follows this overall architecture using stacked
self-attention and point-wise, fully connected layers for both the
encoder and decoder, shown in the left and right halves of Figure 1,
respectively.

Image(filename='images/transformer.png', width=500)

<br>

## Positional Encoding

Since Transformer contains no recurrence and no convolution, in order
for the model to make use of the order of the sequence, we must
inject some information about the relative or absolute position of
the tokens in the sequence.  To this end, we add "positional
encodings" to the input embeddings at the bottoms of the encoder and
decoder stacks.  The positional encodings have the same dimension
$d_{\text{model}}$ as the embeddings, so that the two can be summed.
There are many choices of positional encodings, learned and fixed.

In this work, we use sine and cosine functions of different frequencies:

$$PE_{(pos,2i)} = \sin(pos / 10000^{2i/d_{\text{model}}})$$

$$PE_{(pos,2i+1)} = \cos(pos / 10000^{2i/d_{\text{model}}})$$

where $pos$ is the position and $i$ is the dimension.  That is, each
dimension of the positional encoding corresponds to a sinusoid.  The
wavelengths form a geometric progression from $2\pi$ to $10000 \cdot
2\pi$.  We chose this function because we hypothesized it would
allow the model to easily learn to attend by relative positions,
since for any fixed offset $k$, $PE_{pos+k}$ can be represented as a
linear function of $PE_{pos}$.

We also experimented with using learned positional embeddings instead, and found that the two
versions produced nearly identical results.
We chose the sinusoidal version because it may allow the model to extrapolate
to sequence lengths longer than the ones encountered during training.

In addition, we apply dropout to the sums of the embeddings and the
positional encodings in both the encoder and decoder stacks.  For
the base model, we use a rate of $P_{drop}=0.1$.




In [9]:
def create_fixed_positional_encoding(dim, max_len=5000):
    "Implement the PE function."

    # Compute the positional encodings once in log space.
    pe = torch.zeros(max_len, dim)                      # empty encodings vectors
    position = torch.arange(0, max_len).unsqueeze(1)    # position index

    # $10000^{\frac{2i}{d_{model}}}$
    div_term = torch.exp(
        torch.arange(0, dim, 2) * -(math.log(10000.0) / dim)
    )

    # $PE_{p,2i} = sin\Bigg(\frac{p}{10000^{\frac{2i}{d_{model}}}}\Bigg)$
    pe[:, 0::2] = torch.sin(position * div_term)

    # $PE_{p,2i + 1} = cos\Bigg(\frac{p}{10000^{\frac{2i}{d_{model}}}}\Bigg)$
    pe[:, 1::2] = torch.cos(position * div_term)

    # add batch dimension
    pe = pe.unsqueeze(0).requires_grad_(False)

    return pe   # simple PE (without embedding info)


> Below is an example of the positional encoding which will add in a 
> sine/cosine wave based on position. 
> The frequency and offset of the wave is different for each dimension.

import plotly.express as px

pe = create_fixed_positional_encoding(20, 5000)       # pe: [1, max_seq_len, d_model]

frames = []
for dim in [4, 5, 6, 7]:
    d = {
        'Position': list(range(101)),
        'Embedding': pe[0, :101, dim],
        'Dimension': dim,
    }
    frames.append(pd.DataFrame(d))
df = pd.concat(frames)
fig = px.line(
    df, x="Position", y="Embedding", color="Dimension", title='Positional Encoding', template='none',
)

fig.update_layout(
    width=800, height=400,
    xaxis=dict(
        tickmode='linear',
        tick0=0,
        dtick=5,
        range=[0, 100],
    ),
    yaxis=dict(
        tickmode='linear',
        tick0=0,
        dtick=0.25,
        # range=[-1, 1],
    ),
)

fig.show()


Image(filename='./images/pe.png')

<br>

## Embeddings
Similarly to other sequence transduction models, we use learned
embeddings to convert the input tokens and output tokens to vectors
of dimension $d_{\text{model}}$.  We also use the usual learned
linear transformation and softmax function to convert the decoder
output to predicted next-token probabilities.  In our model, we
share the same weight matrix between the two embedding layers and
the pre-softmax linear transformation, similar to
[(cite)](https://arxiv.org/abs/1608.05859). In the embedding layers,
we multiply those weights by $\sqrt{d_{\text{model}}}$.

In [12]:
# version 2022
class Embeddings(nn.Module):
    def __init__(self, d_model, vocab):
        super().__init__()
        self.lut = nn.Embedding(vocab, d_model)     # lut: lookup table
        self.d_model = d_model
        
    def forward(self, x):
        return self.lut(x) * math.sqrt(self.d_model)

> We can combine PositionalEncoding with Token Embedding:

In [13]:
class EmbeddingsWithPositionalEncoding(nn.Module):
    
    def __init__(self, vocab_size, dim, dropout=0.1, pe_type='fixed', max_len=50000):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, dim)
        self.dim = dim
        
        if pe_type == 'fixed':      # fixed positional encoding
            pe = create_fixed_positional_encoding(dim, max_len)
            self.register_buffer('pe', pe)              # requires_grad=False
        else:                       # learned positional encoding
            self.pe = nn.Parameter(torch.zeros(1, max_len, dim))    # requires_grad=True (defaults to True)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        """
        Args:
            x: Tensor, shape (batch_size, seq_len)
        """
        
        # word/token embedding
        token_embedding = self.embed(x) * math.sqrt(self.dim)     # return shape: (batch_size, seq_len, embed_dim)

        # positional encoding
        positional_encoding = self.pe[:, :x.size(1)]           # return shape: (1, seq_len, embed_dim)
        
        return self.dropout(token_embedding + positional_encoding)

<br>

## LayerNorm

In [14]:
# version 2022
class LayerNorm(nn.Module):
    "Construct a layernorm module (See citation for details)."

    def __init__(self, features, eps=1e-6):
        super().__init__()
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2

In [15]:
class LayerNorm(nn.Module):

    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.bias = nn.Parameter(torch.zeros(dim))
        self.eps = eps
    
    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.weight * (x - mean) / (std + self.eps) + self.bias


# Can be replaced by: `nn.LayerNorm`
# https://pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html

<br>

## Position-wise Feed-Forward Networks

In addition to attention sub-layers, each of the layers in our
encoder and decoder contains a fully connected feed-forward network,
which is applied to each position separately and identically.  This
consists of two linear transformations with a ReLU activation in
between.

$$\mathrm{FFN}(x)=\max(0, xW_1 + b_1) W_2 + b_2$$

While the linear transformations are the same across different
positions, they use different parameters from layer to
layer. Another way of describing this is as two convolutions with
kernel size 1.  The dimensionality of input and output is
$d_{\text{model}}=512$, and the inner-layer has dimensionality
$d_{ff}=2048$.

In [16]:
# version 2022
class FeedForward(nn.Module):
    "Implements FFN equation."

    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.w_2(self.dropout(self.w_1(x).relu()))

In [17]:
class FeedForward(nn.Module):

    def __init__(self, embed_dim, dropout=0.0, bias=True):
        super().__init__()
        self.linear1 = nn.Linear(embed_dim, 4*embed_dim, bias=bias)     # middle layer size is set as 4*embed_dim
        self.linear2 = nn.Linear(4*embed_dim, embed_dim, bias=bias)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        x = self.linear1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

<br>

## Attention

An attention function can be described as mapping a query and a set
of key-value pairs to an output, where the query, keys, values, and
output are all vectors.  The output is computed as a weighted sum of
the values, where the weight assigned to each value is computed by a
compatibility function of the query with the corresponding key.


### Scaled Dot-Product Attention

We call our particular attention "Scaled Dot-Product Attention".
The input consists of queries and keys of dimension $d_k$, and
values of dimension $d_v$.  We compute the dot products of the query
with all keys, divide each by $\sqrt{d_k}$, and apply a softmax
function to obtain the weights on the values.



In practice, we compute the attention function on a set of queries
simultaneously, packed together into a matrix $Q$.  The keys and
values are also packed together into matrices $K$ and $V$.  We
compute the matrix of outputs as:

$$
   \mathrm{Attention}(Q, K, V) = \mathrm{softmax}(\frac{QK^T}{\sqrt{d_k}})V
$$

Image(filename='images/attention.png', width=900)

In [19]:
# version 2022
def attention(query, key, value, mask=None, dropout=None):
    "Compute 'Scaled Dot Product Attention'"
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    p_attn = scores.softmax(dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn


The two most commonly used attention functions are additive
attention, and dot-product
(multiplicative) attention.  Dot-product attention is identical to
our algorithm, except for the scaling factor of
$\frac{1}{\sqrt{d_k}}$. Additive attention computes the
compatibility function using a feed-forward network with a single
hidden layer.  While the two are similar in theoretical complexity,
dot-product attention is much faster and more space-efficient in
practice, since it can be implemented using highly optimized matrix
multiplication code.


While for small values of $d_k$ the two mechanisms perform
similarly, additive attention outperforms dot product attention
without scaling for larger values of $d_k$. We suspect that for
large values of $d_k$, the dot products grow large in magnitude,
pushing the softmax function into regions where it has extremely
small gradients (To illustrate why the dot products get large,
assume that the components of $q$ and $k$ are independent random
variables with mean $0$ and variance $1$.  Then their dot product,
$q \cdot k = \sum_{i=1}^{d_k} q_ik_i$, has mean $0$ and variance
$d_k$.). To counteract this effect, we scale the dot products by
$\frac{1}{\sqrt{d_k}}$.

<br>


### Multi-Head Attention

Multi-head attention allows the model to jointly attend to
information from different representation subspaces at different
positions. With a single attention head, averaging inhibits this.

$$
\mathrm{MultiHead}(Q, K, V) =
    \mathrm{Concat}(\mathrm{head_1}, ..., \mathrm{head_h})W^O \\
    \text{where}~\mathrm{head_i} = \mathrm{Attention}(QW^Q_i, KW^K_i, VW^V_i)
$$

Where the projections are parameter matrices $W^Q_i \in
\mathbb{R}^{d_{\text{model}} \times d_k}$, $W^K_i \in
\mathbb{R}^{d_{\text{model}} \times d_k}$, $W^V_i \in
\mathbb{R}^{d_{\text{model}} \times d_v}$ and $W^O \in
\mathbb{R}^{hd_v \times d_{\text{model}}}$.

In this work we employ $h=8$ parallel attention layers, or
heads. For each of these we use $d_k=d_v=d_{\text{model}}/h=64$. Due
to the reduced dimension of each head, the total computational cost
is similar to that of single-head attention with full
dimensionality.

In [20]:
# version 2022
class MultiHeadAttention(nn.Module):
    def __init__(self, h, d_model, dropout=0.1):
        "Take in model size and number of heads."
        super().__init__()
        assert d_model % h == 0, "embeding dim (d_model) must be divisible by number of heads (h)"
        # We assume d_v always equals d_k
        self.d_k = d_model // h
        self.h = h
        self.attn = None    # record attention score
        self.dropout = nn.Dropout(p=dropout)
        
        # query, key, value (QKV) projections for all heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)

        # output projection
        self.out_proj = nn.Linear(d_model, d_model)


    def forward(self, query, key, value, mask=None):
        "Implements Figure 2"
        if mask is not None:
            # Same mask applied to all h heads.
            mask = mask.unsqueeze(1)
        # nbatches, seq_len, d_model = query.size()     # 【query】的维度为：batch size (nbatches)[batch first], sequence length (seq_len), embedding dimension (d_model)
        nbatches = query.size(0)
        
        # 1) Do all the linear projections in batch from d_model => h x d_k
        query = self.q_proj(query).view(nbatches, -1, self.h, self.d_k).transpose(1, 2) # d_model => h x d_k：实现了 multihead attention（多头注意力机制）
        key = self.k_proj(key).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
        value = self.k_proj(value).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
        
        # 2) Apply attention on all the projected vectors in batch.
        x, self.attn = attention(
            query, key, value, mask=mask, dropout=self.dropout
        )
        
        # 3) "Concat" using a view and apply a final linear.
        x = (
            x.transpose(1, 2)
            .contiguous()
            .view(nbatches, -1, self.h * self.d_k)  # 通过view/reshape，巧妙地实现了concat multihead的操作
        )
        del query
        del key
        del value
        
        return self.out_proj(x)     # 此时返回的变量其维度为：[nbatches, seq_len, d_model]

In [21]:
class MultiHeadAttention(nn.Module):
    
    def __init__(
        self,
        embed_dim,
        num_heads,
        dropout: float = 0.1,
        bias: bool = True,
        ):

        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim (d_model) must be divisible by num_heads"
        self.head_dim = embed_dim // num_heads      # d_k
        self.embed_dim = embed_dim
        self.num_heads = num_heads

        self.attn_score = None    # record attention score
        
        # scaling factor
        self.scaling = self.head_dim**-0.5

        # query, key, value (QKV) projections for all heads
        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=bias)
        self.k_proj = nn.Linear(embed_dim, embed_dim, bias=bias)
        self.v_proj = nn.Linear(embed_dim, embed_dim, bias=bias)

        # output projection
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=bias)

        self.dropout = nn.Dropout(dropout)
        
        # parameter initialization can be done later after the full model is created
        # self.reset_parameters()
    
    # def reset_parameters(self):
    #     nn.init.xavier_normal_(self.q_proj.weight)
    #     nn.init.xavier_normal_(self.k_proj.weight)
    #     nn.init.xavier_normal_(self.v_proj.weight)
    #     nn.init.xavier_uniform_(self.out_proj.weight)
    #     if self.out_proj.bias is not None:
    #         nn.init.constant_(self.out_proj.bias, 0.0)
    
    def forward(
        self, 
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        mask=None,
        return_weights=False
        ):

        bsz, seq_len, embed_dim = query.size()      # batch size (bsz)[batch first], sequence length (query), embedding dimensionality (embed_dim)
        assert embed_dim == self.embed_dim
        assert key.size() == value.size()           # key = value
        
        # 1. Q/K/V projections in batch from d_model => h x d_k
        # calculate query, key, value for all heads in batch 
        # and move head forward to be the batch dim
        q = self.q_proj(query).view(bsz, -1, self.num_heads, self.head_dim).transpose(1, 2)     # bsz, num_heads, seq_len, head_dim
        k = self.k_proj(key).view(bsz, -1, self.num_heads, self.head_dim).transpose(1, 2)       # bsz, num_heads, seq_len, head_dim
        v = self.v_proj(value).view(bsz, -1, self.num_heads, self.head_dim).transpose(1, 2)     # bsz, num_heads, seq_len, head_dim 
        
        # implementation of attention
        # scores = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        scores = (q @ k.transpose(-2, -1)) * self.scaling     # bsz, num_heads, seq_len (query), seq_len (key)
        
        # masked attention
        if mask is not None:
            mask = mask.unsqueeze(1)        # broadcast: same mask applied to all (num_heads) attention heads
            scores = scores.masked_fill(mask == 0, float('-inf'))   # elements with 0 are masked
        
        # attention weight/probability
        attn = F.softmax(scores, dim=-1)    # dim in key sequence
        
        if self.dropout is not None:
            attn = self.dropout(attn)
        
        self.attn_score = attn              # record attention score
        
        # attended/weighted sum
        # (bsz, num_heads, seq_len, seq_len) x (bsz, num_heads, seq_len, head_dim) -> (bsz, num_heads, seq_len, head_dim)
        values = attn @ v
        # values = torch.matmul(attn, v)
        # values = torch.bmm(attn, v)
        
        # combine multihead attn (reshape)
        # (bsz, num_heads, seq_len, head_dim) -> (bsz, seq_len, num_heads, head_dim) -> (bsz, seq_len, embed_dim)
        values = values.transpose(1, 2).reshape(bsz, seq_len, embed_dim)
        # values = values.permute(0, 2, 1, 3).contiguous().view(bsz, seq_len, embed_dim)
        
        # output projection
        out = self.out_proj(values)        # bsz, seq_len, embed_dim
        
        if return_weights:          # return attention weights
            return out, attn
        return out

<br>

### Applications of Attention in our Model

The Transformer uses multi-head attention in three different ways:
1) In "encoder-decoder attention" layers, the queries come from the
previous decoder layer, and the memory keys and values come from the
output of the encoder.  This allows every position in the decoder to
attend over all positions in the input sequence.  This mimics the
typical encoder-decoder attention mechanisms in sequence-to-sequence
models.


2) The encoder contains self-attention layers.  In a self-attention
layer all of the keys, values and queries come from the same place,
in this case, the output of the previous layer in the encoder.  Each
position in the encoder can attend to all positions in the previous
layer of the encoder.


3) Similarly, self-attention layers in the decoder allow each
position in the decoder to attend to all positions in the decoder up
to and including that position.  We need to prevent leftward
information flow in the decoder to preserve the auto-regressive
property.  We implement this inside of scaled dot-product attention
by masking out (setting to $-\infty$) all values in the input of the
softmax which correspond to illegal connections.

<br>

## Encoder and Decoder Stacks

### Encoder

The encoder is composed of a stack of $N=6$ identical layers. 
Each layer has two sub-layers. The first is a multi-head
self-attention mechanism, and the second is a simple, position-wise
fully connected feed-forward network.


We employ a residual connection around each of the two
sub-layers, followed by layer normalization.


That is, the output of each sub-layer is $\mathrm{LayerNorm}(x +
\mathrm{Sublayer}(x))$, where $\mathrm{Sublayer}(x)$ is the function
implemented by the sub-layer itself.  We apply dropout to the
output of each sub-layer, before it is added to the sub-layer input
and normalized.

To facilitate these residual connections, all sub-layers in the
model, as well as the embedding layers, produce outputs of dimension
$d_{\text{model}}=512$.

In [ ]:
# version 2022
class EncoderLayer(nn.Module):
    "Encoder is made up of self-attn and feed forward (defined below)"

    def __init__(self, size, self_attn, feed_forward, dropout):
        super().__init__()
        self.self_attn = self_attn
        self.feed_forward = feed_forward
        self.size = size
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        "Follow Figure 1 (left) for connections."
        # x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask))
        # return self.sublayer[1](x, self.feed_forward)

        # 1. self-attention sublayer
        norm_x = self.norm(x)       # pre-norm, instead of post-norm【这里和文章不一致，先对输入x进行了norm（也称为pre-norm），而文章中是最后才做了norm（也称为post-norm）】
        # [This is inconsistent with the article, the input x is norm (also known as pre-norm) first, and the article is the last norm (also known as post-norm)]
        attn_output = self.self_attn(norm_x, norm_x, norm_x, mask)
        x = x + self.dropout(attn_output)

        # 2. feedforward sublayer
        norm_x = self.norm(x)
        ff_output = self.feed_forward(norm_x)
        return x + self.dropout(ff_output)


In [23]:
class EncoderLayer(nn.Module):
    """
    EncoderLayer consists of self-attention and feed forward layers
    """
    def __init__(self, embed_dim, num_heads, dropout=0.1, pre_norm=True):
        super().__init__()

        self.self_attn = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.ff = FeedForward(embed_dim, dropout)

        self.norm_self_attn = LayerNorm(embed_dim)
        self.norm_ff = LayerNorm(embed_dim)
        
        self.dropout = nn.Dropout(dropout)
        self.pre_norm = pre_norm
    
    def forward(self, x, mask):

        if self.pre_norm:       # pre-norm
            # 1. self-attention sublayer
            norm_x = self.norm_self_attn(x)
            x = x + self.dropout(self.self_attn(norm_x, norm_x, norm_x, mask))
            
            # 2. feedforward sublayer
            norm_x = self.norm_ff(x)
            x = x + self.dropout(self.ff(norm_x))
        else:                   # post-norm
            # 1. self-attention sublayer
            x = x + self.dropout(self.self_attn(x, x, x, mask))
            x = self.norm_self_attn(x)

            # 2. feedforward sublayer
            x = x + self.dropout(self.ff(x))
            x = self.norm_ff(x)
        
        return x

> The encoder is composed of a stack of $N=6$ encoder layers.

In [24]:
# version 2022
def clones(module, N):
    "Produce N identical layers."
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

class Encoder(nn.Module):
    "Core encoder is a stack of N layers"

    def __init__(self, layer, N):
        super(Encoder, self).__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)

    def forward(self, x, mask):
        "Pass the input (and mask) through each layer in turn."
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

In [25]:

class Encoder(nn.Module):
    """
    Encoder consists of multiple EncoderLayer sublayers
    """

    def __init__(self, embed_dim, num_layers, num_heads, dropout=0.1, pre_norm=True):
        super().__init__()
        
        # 先实例化，再deepcopy
        # instantiated once, and deepcopy N times
        encoder_layer = EncoderLayer(embed_dim, num_heads, dropout, pre_norm)
        self.layers = nn.ModuleList([copy.deepcopy(encoder_layer) for _ in range(num_layers)])  # deepcopy is required
        
        # 或者，直接实例化N次
        # or instantiated N times directly
        # self.layers = nn.ModuleList([EncoderLayer(embed_dim, num_heads, dropout, pre_norm) for _ in range(num_layers)])  # no need to deepcopy

        self.norm = LayerNorm(embed_dim)
    
    def forward(self, x, mask):
        
        for layer in self.layers:
            x = layer(x, mask)

        return self.norm(x)

<br>

### Decoder

The decoder is also composed of a stack of $N=6$ identical layers.



In addition to the two sub-layers in each encoder layer, the decoder
inserts a third sub-layer, which performs multi-head attention over
the output of the encoder stack.  Similar to the encoder, we employ
residual connections around each of the sub-layers, followed by
layer normalization.

In [26]:
# version 2022
class DecoderLayer(nn.Module):
    "Decoder is made of self-attn, src-attn, and feed forward (defined below)"

    def __init__(self, size, self_attn, src_attn, feed_forward, dropout):
        super().__init__()
        self.size = size
        self.self_attn = self_attn
        self.src_attn = src_attn
        self.feed_forward = feed_forward
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, memory, src_mask, tgt_mask):
        "Follow Figure 1 (right) for connections."

        # 1. masked self-attention sublayer【对decoder的input做self-attention (self_attn)】
        norm_x = self.norm(x)   # pre-norm, instead of post-norm【这里和文章不一致，先对输入x进行了norm（也称为pre-norm），而文章中是最后才做了norm（也称为post-norm）】
        attn_output = self.self_attn(norm_x, norm_x, norm_x, tgt_mask)  # 此处的 mask 为tagt_mask（target mask，即decoder输入端的mask）
        x = x + self.dropout(attn_output)

        # 2. cross-attention sublayer【decoder的input 与 encoder的输出 做 cross-attention (src_attn)】
        # memory 即为 encoder 的输出
        norm_x = self.norm(x)               # pre-norm, instead of post-norm
        attn_output = self.src_attn(norm_x, memory, memory, src_mask)   # 此处的 mask 为src_mask（source mask，即 mask encoder端的padding）
                                            # Q = norm_x, K/V = memory
        x = x + self.dropout(attn_output)
        
        # 3. feedforward sublayer
        norm_x = self.norm(x)
        ff_output = self.feed_forward(norm_x)
        return x + self.dropout(ff_output)

In [27]:
class DecoderLayer(nn.Module):
    """
    DecoderLayer consists of self attention, cross attention, and feed forward
    """
    def __init__(self, embed_dim, num_heads, dropout=0.1, pre_norm=True):
        super().__init__()
        
        self.self_attn = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.ff = FeedForward(embed_dim, dropout)

        self.norm_self_attn = LayerNorm(embed_dim)
        self.norm_cross_attn = LayerNorm(embed_dim)
        self.norm_ff = LayerNorm(embed_dim)

        self.dropout = nn.Dropout(dropout)
        self.pre_norm = pre_norm
    
    def forward(self, x, memory, src_mask, tgt_mask):   # caution: the order of args should be consistent across modules
        """
        x, tgt_mask : decoder input and associated 'causal' mask (auto-regressive)
        memory, src_mask : encoder output (serves as K, V in cross attention) and associated mask
        """
        
        if self.pre_norm:       # pre-norm
            # 1. masked self-attention sublayer
            norm_x = self.norm_self_attn(x)
            x = x + self.dropout(self.self_attn(norm_x, norm_x, norm_x, tgt_mask))
            
            # 2. cross-attention sublayer
            norm_x = self.norm_cross_attn(x)
            x = x + self.dropout(self.cross_attn(norm_x, memory, memory, src_mask)) # decoder side 'queries' encoder output (memory)
            
            # 3. feedforward sublayer
            norm_x = self.norm_ff(x)
            x = x + self.dropout(self.ff(norm_x))
        else:                   # post-norm
            # 1. masked self-attention sublayer
            x = x + self.dropout(self.self_attn(x, x, x, tgt_mask))
            x = self.norm_self_attn(x)

            # 2. cross-attention sublayer
            x = x + self.dropout(self.cross_attn(x, memory, memory, src_mask))      # decoder side 'queries' encoder output (memory)
            x = self.norm_cross_attn(x)
            
            # 3. feedforward sublayer
            x = x + self.dropout(self.ff(x))
            x = self.norm_ff(x)
        
        return x

> The decoder is also composed of a stack of $N=6$ decoder layers.

In [28]:
# version 2022
class Decoder(nn.Module):
    "Generic N layer decoder with masking."

    def __init__(self, layer, N):
        super(Decoder, self).__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)
    
    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)

In [29]:
class Decoder(nn.Module):
    """
    Decoder consists of multiple DecoderLayer sublayers
    """

    def __init__(self, embed_dim, num_layers, num_heads, dropout=0.1, pre_norm=True):
        super().__init__()

        # # 先实例化，再deepcopy N次
        # Instantiated once, and deepcopy N times
        # decoder_layer = DecoderLayer(embed_dim, num_heads, dropout, pre_norm)
        # self.layers = nn.ModuleList([copy.deepcopy(decoder_layer) for _ in range(num_layers)])  # 需要deepcopy，否则会导致 所有层共享同一组参数，无法实现独立训练

        # 或者，直接实例化N次
        # or instantiated N times directly
        self.layers = nn.ModuleList([DecoderLayer(embed_dim, num_heads, dropout, pre_norm) for _ in range(num_layers)])
        self.norm = LayerNorm(embed_dim)

    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)

        return self.norm(x)


We also modify the self-attention sub-layer in the decoder stack to
prevent positions from attending to subsequent positions.  This
masking, combined with fact that the output embeddings are offset by
one position, ensures that the predictions for position $i$ can
depend only on the known outputs at positions less than $i$.

In [30]:
def create_causal_mask(size):
    "Mask out subsequent positions to preserve the auto-regressive property."
    attn_shape = (1, size, size)
    causal_mask = torch.triu(torch.ones(attn_shape), diagonal=1).type(torch.uint8)
    return causal_mask == 0


> Below the attention mask shows the position each target word (row) is
> allowed to look at (column). 
> To preserve the auto-regressive property, words are blocked for attending to
> future words during training.

import plotly.graph_objects as go

n = 21
matrix = create_causal_mask(n).int().numpy()[0]

heat = go.Heatmap(
        z=matrix,
        # x=xlabels,
        # y=ylabels,
        xgap=1, ygap=1,
        colorscale='PuBu',
        colorbar_thickness=20,
        colorbar_ticklen=3,
    )
layout = go.Layout(
    title_text="Causal Mask", 
    title_x=0.5, 
    width=400, height=400,
    yaxis_autorange='reversed', 
    xaxis=dict(
        ticks='outside'
    ),
    yaxis=dict(
        ticks='outside'
    ),
)
fig=go.Figure(data=[heat], layout=layout)
fig.show()

Image(filename='images/causal_mask.png')

<br>

## Transformer Model

> Transformer model consists of Encoder, Decoder, and a final projection layer


In [33]:
class Generator(nn.Module):
    """
    Define decoder side lanaguge model head (`final_proj`, a linear projection) and softmax for generation.
    Note: To tie the weights of final_proj with nn.Embedding, bias term in final_proj is set to be False.
    """
    
    def __init__(self, embed_dim, vocab_size):
        super().__init__()
        self.final_proj = nn.Linear(embed_dim, vocab_size, bias=False)    # projection to vocab size
    
    def forward(self, x):
        return F.log_softmax(self.final_proj(x), dim=-1)      # softmax and log

In [ ]:
class Transformer(nn.Module):
    """
    Transformer consists of Encoder, Decoder, and a final projection layers
    """
    def __init__(self, src_vocab_size, tgt_vocab_size, embed_dim, num_layers, num_heads, dropout=0.1, pre_norm=True, pe_type='fixed'):
        super().__init__()
        
        self.src_embed = EmbeddingsWithPositionalEncoding(src_vocab_size, embed_dim, dropout, pe_type)
        self.tgt_embed = EmbeddingsWithPositionalEncoding(tgt_vocab_size, embed_dim, dropout, pe_type)
        
        self.encoder = Encoder(embed_dim, num_layers, num_heads, dropout, pre_norm)
        self.decoder = Decoder(embed_dim, num_layers, num_heads, dropout, pre_norm)
        
        self.generator = Generator(embed_dim, tgt_vocab_size)

        # post init
        self.post_init()
        
        
    def encode(self, src, src_mask):
        """Encoder process"""
        x = self.src_embed(src)                     # embedding of src
        enc_out = self.encoder(x, src_mask)         # encoder output
        return enc_out
    
    def decode(self, tgt, memory, src_mask, tgt_mask):  # the order of args should be consistent across modules
        """Decoder process"""
        x = self.tgt_embed(tgt)                     # embedding of tgt
        dec_out = self.decoder(x, memory, src_mask, tgt_mask)    # the 'encoder output' serves as K,V in 'decoder cross sttention'
        return dec_out
        
    def forward(self, src, tgt, src_mask, tgt_mask):
        memory = self.encode(src, src_mask)                      # encoder output (memory)
        dec_out = self.decode(tgt, memory, src_mask, tgt_mask)   # decoder output
        return dec_out
    
        # out = self.generator(dec_out)                            # final layer projection
        
    
    def post_init(self):
        "Tie final_proj weight with target embedding weight (defaults to do so)"
        self.generator.final_proj.weight = self.tgt_embed.embed.weight
        # print("Source (encoder) and target (decoder) embedding weights are not tied by default.")

    def tie_weights(self):
        """
        Tie source and target embedding weights when necessary.
        For example, tie weights if source and target embeddings use the same vocabulary.
        """
        self.src_embed.embed.weight = self.tgt_embed.embed.weight
        print("Source (encoder) and target (decoder) embedding weights are now tied.")
        
    @property
    def device(self) -> torch.device:
        """
        `torch.device`: The device on which the module is.
        (assuming that all the module parameters are on the same device)
        """
        return next(self.parameters()).device

<br>

<br>

## Create Full Model

> Here we define a function from hyperparameters to a full model.

In [35]:
def create_model(
    src_vocab_size, 
    tgt_vocab_size, 
    embed_dim=512, 
    num_layers=6,
    num_heads=8, 
    dropout=0.1, 
    pre_norm=True, 
    pe_type='fixed',
    device=None,
):
    """
    config: 
        configurations to set the model
        
        - src_vocab_size
        - tgt_vocab_size
        
        - embed_dim
        - num_layers
        - num_heads
        - dropout
        - pre_norm
        - pe_type

        - device
    """
    
    model = Transformer(src_vocab_size, tgt_vocab_size, embed_dim, num_layers, num_heads, dropout, pre_norm, pe_type)
    
    # Initialize model weights
    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
    
    if device is not None:      # note: `0` indicates `cuda:0`
        model = model.to(device)
    
    return model

<br>

## Inference Test

#### Use simple cases to quickly test if the implemented code is executable

> Here we make a forward step to generate a prediction of the
model. We try to use our transformer to memorize the input. As you
will see the output is randomly generated due to the fact that the
model is not trained yet. In the next tutorial we will build the
training function and try to train our model to memorize the numbers
from 1 to 10.

In [ ]:
def inference_test():
    test_model = create_model(
        src_vocab_size=11, 
        tgt_vocab_size=11, 
        embed_dim=512, 
        num_layers=2, 
        num_heads=8, 
        dropout=0.1, 
        pre_norm=True, 
        pe_type='fixed',
        device=None,
    )
    # do not tie the weight for a random test:
    test_model.generator.final_proj.weight = nn.Parameter(torch.randn_like(test_model.generator.final_proj.weight))

    test_model.eval()
    src = torch.LongTensor([[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]])
    src_mask = torch.ones(1, 1, 10)

    memory = test_model.encode(src, src_mask)
    ys = torch.zeros(1, 1).type_as(src)     # note: ys[0]=0, i.e., ys starts with 0
    
    # model rollout
    for i in range(9):
        tgt_mask = create_causal_mask(ys.size(1)).type_as(src.data)
        out = test_model.decode(ys, memory, src_mask, tgt_mask)
        prob = test_model.generator(out[:, -1])     # last token
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.data[0]
        ys = torch.cat(
            [ys, torch.empty(1, 1).type_as(src.data).fill_(next_word)], dim=1
        )

    print("Example Untrained Model Prediction:", ys)





for _ in range(10):
    inference_test()